# Good Culture in AI answer engines

ChatGPT, Gemini and Perplexity were asked the same questions and recorded **which brands they name, which one they tell the shopper to buy, and what they cite.** 

## 0. Setup

In [33]:
import os, json, re, warnings, itertools, textwrap
from pathlib import Path
import numpy as np, pandas as pd, yaml
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# ---------- settings ----------
EVAL_DIR = Path(os.environ.get("GC_EVAL_DIR", "evaluations/good_culture"))
WAVE     = os.environ.get("GC_WAVE", "2026_Q4")
DRAWS    = 2000       # re-draws used for the 95% ranges
SEED     = 42
MIN_N    = 8          # fewer answers than this -> "too few answers", no range
USE_REGEX_MENTIONS = True   # brand counts as named if the AI judge OR the text matcher found it
# -------------------------------

WAVE_DIR = EVAL_DIR / "results" / WAVE
TABLES   = WAVE_DIR / "tables";  TABLES.mkdir(parents=True, exist_ok=True)
FIGS     = WAVE_DIR / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
profile  = yaml.safe_load(open(EVAL_DIR / "brand_profile.yaml", encoding="utf-8"))
learned  = yaml.safe_load(open(EVAL_DIR / "learned.yaml")) if (EVAL_DIR / "learned.yaml").exists() else {}
TARGET   = profile["target"]
LEADER   = profile["market_leader"] if profile["market_leader"] != TARGET else next(b for b in profile["brands"] if b != TARGET)
BRANDS   = list(dict.fromkeys(list(profile["brands"]) + list((learned or {}).get("brands") or {})))
slug     = lambda s: re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")[:40]
BRAND_BY_SLUG = {slug(b): b for b in BRANDS}
WEB      = "default"

# Main comparison set used for EVERY chart and finding (brands not in it are not charted)
BRANDS_7 = ["Good Culture", "Daisy", "Hood", "Nancy's", "Breakstone's", "Lactaid", "Knudsen"]

# learned.yaml files some brand-owned sites under "retailer"; correct that for the citation test
OWNED_DOMAINS = {"goodculture.com": TARGET, "daisybrand.com": "Daisy", "hood.com": "Hood", "knudsen.com": "Knudsen",
                 "nancysyogurt.com": "Nancy's", "kalonasupernatural.com": "Kalona SuperNatural", "prairiefarms.com": "Prairie Farms",
                 "greenvalleylactosefree.com": "Green Valley Creamery", "friendshipdairies.com": "Friendship Dairies",
                 "breakstones.com": "Breakstone's", "organicvalley.coop": "Organic Valley"}

UNBRANDED   = ["discovery", "attributes", "personas"]
STAGE_LABEL = {"discovery": "Broad / discovery searches", "attributes": "Feature-led searches", "personas": "Shopper-segment searches"}
ENGINE_LABEL = {"chatgpt": "ChatGPT", "gemini": "Gemini", "perplexity": "Perplexity"}
eng_name = lambda e: ENGINE_LABEL.get(e, e)
GREEN, GREY, DARK, LIGHT = "#2E7D32", "#B7BEC6", "#5F6B76", "#E6E9ED"
color  = lambda b: GREEN if b == TARGET else GREY
pretty = lambda pid: str(pid).split("__")[-1].replace("_", " ")
pct    = lambda v: f"{v:.0%}"
pts    = lambda v: f"{v*100:+.0f} pts"
FINDINGS = []     # filled by each section, shown in the summary

BAR_NOTE = f"95% range across questions."

def _title(title, subtitle=""):
    t = f"<b>{title}</b>"
    if subtitle: t += f"<br><span style='font-size:13px;color:#6b7785'>{subtitle}</span>"
    return dict(text=t, x=0, xanchor="left", y=0.97, yanchor="top", font=dict(size=18, color="#1f2933"))

def _finish(fig, title, subtitle, note, height, width=1150, top=120):
    lines = (note.count("<br>") + 1) if note else 0
    bot = (44 + 16 * lines) if note else 30
    fig.update_layout(template="simple_white", title=_title(title, subtitle), showlegend=False, height=height, width=width,
                        title_y=0.95,
                        title_yanchor="top",
                        font=dict(family="Arial, Helvetica, sans-serif", size=12, color="#333"),
                        margin=dict(l=10, r=20, t=top, b=bot), plot_bgcolor="white", paper_bgcolor="white")
    if note:
        plot_h = max(height - top - bot, 1)
        fig.add_annotation(text=note, xref="paper", yref="paper", x=0, y=-34 / plot_h, yanchor="top", showarrow=False,
                           xanchor="left", align="left", font=dict(size=11, color="#7b8794"))
    return fig

_png_ok = [True]
def show(fig, name):
    # fig.write_html(FIGS / f"{name}.html", include_plotlyjs="cdn")
    if _png_ok[0]:
        try: fig.write_image(FIGS / f"{name}.png", scale=2)
        except Exception as e: _png_ok[0] = False; print("(PNG export unavailable; HTML copies are saved in figures/. Reason:", str(e)[:80], ")")
    fig.show()

def hbar_panels(
    panels, fmt, title, subtitle="", note=BAR_NOTE,
    spacing=0.12, top_n=7, order_by="first", xtickfmt=".0%"
):
    """order_by: 'first' = first panel; 'mean' = mean across panels."""
    names = list(panels)
    n = len(names)
    fig = make_subplots(rows=1, cols=n, subplot_titles=names,
                        horizontal_spacing=spacing)

    # Establish common brand hierarchy
    if order_by == "first":
        anchor_df = panels[names[0]].dropna(subset=["value"])
        order_df = anchor_df.groupby("brand", observed=True)["value"].first()

    elif order_by == "mean":
        combined = pd.concat(
            [panels[nm][["brand", "value"]] for nm in names],
            ignore_index=True
        )
        order_df = combined.groupby("brand", observed=True)["value"].mean()

    else:
        raise ValueError("order_by must be 'first' or 'mean'")

    global_order = order_df.sort_values().tail(top_n).index.tolist()
    rows_max = len(global_order)

    for i, nm in enumerate(names, 1):
        d = panels[nm].dropna(subset=["value"])
        d["brand"] = d["brand"].astype(str)
        d = d.set_index("brand").reindex(global_order).reset_index()
        d["flag"] = d["flag"].fillna("") if "flag" in d else ""

        # Don't turn genuinely missing observations into zero
        d["value"] = d["value"].fillna(0)

        lo, hi = d.lo.fillna(d.value), d.hi.fillna(d.value)
        right_edge = np.maximum(d.value.values, hi.values)

        err = dict(
            type="data",
            symmetric=False,
            array=(hi - d.value).values,
            arrayminus=(d.value - lo).values,
            color="#333",
            thickness=1.2,
            width=3
        )

        fig.add_trace(
            go.Bar(
                x=d.value, y=d.brand, orientation="h",
                marker_color=[color(b) for b in d.brand],
                error_x=err, width=0.65,
                hovertemplate="%{y}: %{x:.3f}<extra></extra>"
            ),
            1, i
        )

        pad = d.value.max() * 0.04

        labs = [
            ("<b>" + fmt(v) + "</b>" if b == TARGET else fmt(v))
            + ("*" if f else "")
            for v, b, f in zip(
                d.value, d.brand, d.get("flag", [""] * len(d))
            )
        ]

        fig.add_trace(
            go.Scatter(
                x=right_edge + pad,
                y=d.brand,
                mode="text",
                text=labs,
                textposition="middle right",
                textfont=dict(
                    size=12,
                    color=[GREEN if b == TARGET else "#555" for b in d.brand]
                ),
                hoverinfo="skip"
            ),
            1, i
        )

        fig.update_yaxes(
            tickvals=list(d.brand),
            ticktext=[
                f"<b style='color:{GREEN}'>{b}</b>"
                if b == TARGET else b
                for b in d.brand
            ],
            showline=False, ticks="", showgrid=False,
            row=1, col=i
        )

        max_bound = (right_edge + pad).max()
        fig.update_xaxes(
            visible=True, showline=True, linecolor="#B7BEC6", ticks="outside",
            tickformat=xtickfmt, tickfont=dict(size=11, color="#5F6B76"),
            showgrid=False, zeroline=False,
            range=[0, max_bound * 1.15],
            row=1, col=i
        )

    fig.update_annotations(font=dict(size=14, color="#333", family="Arial"))

    return _finish(
        fig, title, subtitle, note,
        height=200 + 45 * rows_max,
        width=800
    )

## 1. Data check

In [23]:
plan = pd.read_csv(WAVE_DIR / "plan.csv")
ans = pd.DataFrame([{k: json.loads(l).get(k) for k in ["run_id", "n_turns"]} for l in open(WAVE_DIR / "answers.jsonl", encoding="utf-8") if l.strip()]).drop_duplicates("run_id")
runs = plan.drop(columns=["n_turns"]).merge(ans, on="run_id")
qpath = TABLES / "quality_report.csv"
excluded = set(pd.read_csv(qpath).query("excluded")["run_id"]) if qpath.exists() else set()
runs = runs[~runs.run_id.isin(excluded)]
n_off = int((runs.web_search != WEB).sum()); runs = runs[runs.web_search == WEB].copy()

J = pd.read_csv(TABLES / "judgments.csv")
J["known"]   = J["known"].astype("boolean").fillna(False).astype(bool)
J["refusal"] = J["refusal"].astype("boolean").fillna(False).astype(bool)
J = J[J.run_id.isin(runs.run_id)]
M = pd.read_csv(TABLES / "mentions.csv") if (TABLES / "mentions.csv").exists() else pd.DataFrame(columns=["run_id", "turn", "brand", "list_rank"])
M = M.merge(runs[["run_id", "n_turns"]], on="run_id"); M = M[M.turn == M.n_turns]
C = pd.read_csv(TABLES / "citations.csv") if (TABLES / "citations.csv").exists() else pd.DataFrame()

runs["judged"]  = runs.run_id.isin(J.run_id)
runs["refused"] = runs.run_id.isin(J.loc[J.refusal, "run_id"])
runs["swapped"] = runs.prompt_id.str.endswith("__swapped")

core = runs[runs.group == "core"].assign(section=lambda d: d.stage.map(lambda s: STAGE_LABEL.get(s, s)))
t = core.pivot_table(index="section", columns="engine", values="run_id", aggfunc="count", margins=True, margins_name="Total").fillna(0).astype(int)
t["Distinct questions"] = core.groupby("section").prompt_id.nunique().reindex(t.index); display(t)
q = J[J.brand.notna()].quote_verified.astype(bool).mean()
print(f"AI-judge quotes found word-for-word in the answer: {q:.0%} | answers the judge could not score: {(~runs.judged).sum()} | refusals: {runs.refused.sum()} | dropped by quality check: {len(excluded)} | web-search-off runs set aside: {n_off}")
print("All ranges treat the QUESTION as the unit, so few distinct questions per section -> wide ranges.")
if not C.empty:
    cited = runs[runs.run_id.isin(C.run_id)].groupby("engine").size() / runs.groupby("engine").size()
    print("Share of answers that cite at least one web source (evidence the engines are searching live):", ", ".join(f"{eng_name(e)} {v:.0%}" for e, v in cited.items()))

engine,chatgpt,gemini,perplexity,Total,Distinct questions
section,,,,,
Broad / discovery searches,10,10,10,30,5.0
Feature-led searches,18,18,18,54,9.0
Shopper-segment searches,10,10,10,30,5.0
comparisons,8,8,8,24,8.0
objections,5,5,5,15,5.0
Total,51,51,51,153,NaN


AI-judge quotes found word-for-word in the answer: 100% | answers the judge could not score: 0 | refusals: 0 | dropped by quality check: 0 | web-search-off runs set aside: 0
All ranges treat the QUESTION as the unit, so few distinct questions per section -> wide ranges.
Share of answers that cite at least one web source (evidence the engines are searching live): ChatGPT 90%, Gemini 88%, Perplexity 90%


## 2. Calculation methodology

For each answer-brand pair, the pipeline records whether the brand was mentioned, whether it was identified as a top pick, its list rank when available, and sentiment when available. Metrics are first aggregated at the question level across engines and repeated runs, then averaged across questions within each section. This prevents questions with more repeated runs from receiving greater weight.

The overall result is the average of the three sections: open questions, feature searches, and shopper types.


In [24]:
METRICS = ["mention", "top_pick", "avg_rank", "sentiment", "sov"]

def present_sets(R):
    """(run_id, brand) pairs where the brand is named (AI judge, plus the text matcher if enabled)."""
    jk = J[J.run_id.isin(R.run_id) & J.known & J.brand.notna()]
    present = set(zip(jk.run_id, jk.brand))
    if USE_REGEX_MENTIONS:
        mm = M[M.run_id.isin(R.run_id)]; present |= set(zip(mm.run_id, mm.brand))
    return present

def build_matrices(R):
    Jr = J[J.run_id.isin(R.run_id)]; jk = Jr[Jr.known & Jr.brand.notna()]
    present = present_sets(R)
    unk = Jr[~Jr.known & Jr.brand.notna()].assign(b=lambda d: d.brand.str.lower().str.strip()).groupby("run_id").b.nunique()
    known_n = pd.Series([r for r, _ in present]).value_counts()
    n_brands = np.maximum((known_n.reindex(R.run_id).fillna(0) + unk.reindex(R.run_id).fillna(0)).values, 1)
    role = jk.set_index(["run_id", "brand"]).role.to_dict()
    sent = jk.groupby(["run_id", "brand"]).sentiment.mean().to_dict()
    rank = M.groupby(["run_id", "brand"]).list_rank.min().to_dict()
    idx = R.run_id.values; num, den = {}, {}
    for b in BRANDS:
        pr  = np.array([(r, b) in present for r in idx]); top = np.array([role.get((r, b)) == "top_pick" for r in idx])
        rk  = np.array([rank.get((r, b), np.nan) for r in idx], float); se = np.array([sent.get((r, b), np.nan) for r in idx], float)
        rk_ok, se_ok = pr & ~np.isnan(rk), pr & ~np.isnan(se)
        num[(b, "mention")],   den[(b, "mention")]   = pr.astype(float),  np.ones(len(idx))
        num[(b, "top_pick")],  den[(b, "top_pick")]  = top.astype(float), np.ones(len(idx))
        num[(b, "avg_rank")],  den[(b, "avg_rank")]  = np.where(rk_ok, rk, 0), rk_ok.astype(float)
        num[(b, "sentiment")], den[(b, "sentiment")] = np.where(se_ok, se, 0), se_ok.astype(float)
        num[(b, "sov")],       den[(b, "sov")]       = pr.astype(float), n_brands
    cols = pd.MultiIndex.from_tuples(list(num))
    return pd.DataFrame(num, index=idx).set_axis(cols, axis=1), pd.DataFrame(den, index=idx).set_axis(cols, axis=1)

ELIG = runs[(runs.group == "core") & runs.stage.isin(UNBRANDED) & runs.judged & ~runs.refused].reset_index(drop=True)
NUMM, DENM = build_matrices(ELIG)
COLS = list(NUMM.columns); COLIX = {c: i for i, c in enumerate(COLS)}; NUMA, DENA = NUMM.values, DENM.values

def _by_question(num, den, lab, P):
    sn = np.zeros((P, num.shape[1])); sd = np.zeros_like(sn)
    np.add.at(sn, lab, num); np.add.at(sd, lab, den)
    with np.errstate(invalid="ignore", divide="ignore"): return sn / sd
def _avg(r):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning); return np.nanmean(r, axis=0)

def estimate(rows, cols, B=0, seed=SEED):
    lab, uniq = pd.factorize(ELIG.prompt_id.values[rows]); P = len(uniq)
    num, den = NUMA[rows][:, cols], DENA[rows][:, cols]
    point, n = _avg(_by_question(num, den, lab, P)), den.sum(axis=0); draws = None
    if B:
        rng = np.random.default_rng(seed); groups = [np.where(lab == p)[0] for p in range(P)]; draws = np.empty((B, len(cols)))
        for i in range(B):
            sel, l2 = [], []
            for j, p in enumerate(rng.integers(0, P, P)):
                g = groups[p]; s = g[rng.integers(0, len(g), len(g))]; sel.append(s); l2.append(np.full(len(s), j))
            sel, l2 = np.concatenate(sel), np.concatenate(l2); draws[i] = _avg(_by_question(num[sel], den[sel], l2, P))
    return point, n, draws

def rng95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return (np.nan, np.nan) if len(x) < 50 else tuple(np.percentile(x, [2.5, 97.5]))

# Which brands get charted? Every brand the engines actually name in at least MIN_N answers. Nobody is hand-picked.
named_total = pd.Series({b: NUMM[(b, "mention")].sum() for b in BRANDS}).sort_values(ascending=False)
VIS = [b for b in named_total.index if named_total[b] >= MIN_N or b == TARGET]
print(f"{len(ELIG)} answers scored. {len(VIS)} of {len(BRANDS)} tracked brands are named in at least {MIN_N} answers and are charted; the rest are in the appendix table.")
print("Charted brands:", ", ".join(VIS))
BRANDS_CMP = [b for b in BRANDS_7 if b in VIS]      # main comparison set that actually has enough data

def missing_note(shown, reason=None, default="no data"):
    """Short footnote for any BRANDS_7 brand absent from a chart, with the reason."""
    shown, groups = set(map(str, shown)), {}
    for b in BRANDS_7:
        if b in shown: continue
        if b not in BRANDS: r = "not tracked"
        elif reason: r = reason
        elif b not in VIS: r = f"named in <{MIN_N} answers"
        else: r = default
        groups.setdefault(r, []).append(b)
    return ("* Not shown: " + "; ".join(f"{', '.join(v)} ({r})" for r, v in groups.items()) + ".") if groups else ""

def with_missing(note, miss):
    return "<br>".join(x for x in (note, miss) if x)

def grade(lo, hi, n, b=None):
    if n < MIN_N or np.isnan(lo): return "low (too few answers)"
    return "high confidence" if (lo > 0 or hi < 0) else "directional (range includes zero)"


114 answers scored. 14 of 22 tracked brands are named in at least 8 answers and are charted; the rest are in the appendix table.
Charted brands: Good Culture, Daisy, Nancy's, Hood, Breakstone's, store brand, Lactaid, Kalona SuperNatural, Great Value, Trader Joe's, Prairie Farms, Organic Valley, Friendship Dairies, MULU


In [27]:
# score every (engine, section) cell once, with 95% ranges for the charted brands
engines = sorted(ELIG.engine.unique()); CI_COLS = [COLIX[(b, m)] for b in VIS for m in METRICS]; CIPOS = {c: k for k, c in enumerate(CI_COLS)}
STORE, POINT, NSUM, NROWS, NAMED, rows_out = {}, {}, {}, {}, {}, []
cell_id = 0
for stage in UNBRANDED:
    for eng in engines + ["ALL"]:
        rws = np.where((ELIG.stage == stage) & ((ELIG.engine == eng) if eng != "ALL" else True))[0]
        if len(rws) == 0: continue
        cell_id += 1
        pt, n, _ = estimate(rws, list(range(len(COLS)))); _, _, dr = estimate(rws, CI_COLS, B=DRAWS, seed=SEED + cell_id)
        STORE[(eng, stage)] = dr; POINT[(eng, stage)] = pt; NSUM[(eng, stage)] = n; NROWS[(eng, stage)] = len(rws)
        NAMED[(eng, stage)] = NUMA[rws].sum(axis=0)
        for j, (b, m) in enumerate(COLS):
            n_sc = int(n[j]) if m in ("avg_rank", "sentiment") else len(rws)
            lo, hi = (rng95(dr[:, CIPOS[j]]) if (j in CIPOS and n_sc >= MIN_N) else (np.nan, np.nan))
            rows_out.append(dict(stage=stage, engine=eng, brand=b, metric=m, value=pt[j], ci_lo=lo, ci_hi=hi, n_answers=n_sc, flag="*" if n_sc < MIN_N else ""))
scoreboard = pd.DataFrame(rows_out); scoreboard.to_csv(TABLES / "scoreboard.csv", index=False)

def cells(eng): return [s for s in UNBRANDED if (eng, s) in STORE]
def ov_point(eng, cols): return _avg(np.array([POINT[(eng, s)][cols] for s in cells(eng)]))
def ov_draws(eng, cols):
    ks = [CIPOS[c] for c in cols]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning); return np.nanmean(np.array([STORE[(eng, s)][:, ks] for s in cells(eng)]), axis=0)

# OV[(engine, brand, metric)] = (value, lo, hi, n, flag).  "Overall" = average of the three sections.
OV = {}
for eng in engines + ["ALL"]:
    for b in VIS:
        for m in METRICS + ["funnel"]:
            if m == "funnel":
                ct, cm = COLIX[(b, "top_pick")], COLIX[(b, "mention")]
                p = ov_point(eng, [ct, cm]); val = p[0] / p[1] if p[1] > 0 else np.nan
                n = int(sum(NAMED[(eng, s)][cm] for s in cells(eng)))
                d = ov_draws(eng, [ct, cm]); 
                with np.errstate(invalid="ignore", divide="ignore"): ratio = d[:, 0] / d[:, 1]
                lo, hi = rng95(ratio) if n >= MIN_N else (np.nan, np.nan)
            else:
                c = COLIX[(b, m)]; val = ov_point(eng, [c])[0]
                n = int(sum((NSUM[(eng, s)][c] if m in ("avg_rank", "sentiment") else NROWS[(eng, s)]) for s in cells(eng)))
                lo, hi = rng95(ov_draws(eng, [c])[:, 0]) if n >= MIN_N else (np.nan, np.nan)
            OV[(eng, b, m)] = (val, lo, hi, n, "*" if n < MIN_N else "")

def board(eng, metric, brands=None):
    return pd.DataFrame([dict(brand=b, value=OV[(eng, b, metric)][0], lo=OV[(eng, b, metric)][1], hi=OV[(eng, b, metric)][2],
                              n=OV[(eng, b, metric)][3], flag=OV[(eng, b, metric)][4]) for b in (brands or BRANDS_CMP)]).dropna(subset=["value"])
def stage_board(stage, metric, eng="ALL"):
    d = scoreboard.query("engine==@eng and stage==@stage and metric==@metric and brand in @BRANDS_CMP").dropna(subset=["value"])
    return d.rename(columns={"ci_lo": "lo", "ci_hi": "hi"})[["brand", "value", "lo", "hi", "flag"]]
def paired(eng, metric, a, b):
    ca, cb = COLIX[(a, metric)], COLIX[(b, metric)]; d = ov_draws(eng, [ca, cb]); p = ov_point(eng, [ca, cb])
    n = min(OV[(eng, a, metric)][3], OV[(eng, b, metric)][3]); lo, hi = rng95(d[:, 0] - d[:, 1]) if n >= MIN_N else (np.nan, np.nan)
    return p[0] - p[1], lo, hi, n
def rank_of(d, brand=TARGET):
    v = d.set_index("brand").value; return int((v > v[brand]).sum() + 1), len(v)
def best_other(d):
    o = d[d.brand != TARGET].sort_values("value", ascending=False); return o.iloc[0] if len(o) else None
print("scored.")

scored.


## 3. Unbranded search visibility

**Measured as:** % of unbranded AI search answers that mention each brand, shown by question type and engine.

**Question:** whether Good Culture is consistently surfaced in AI-generated answers when the shopper does not specify a brand.


In [34]:
d = board("ALL", "mention"); r, n = rank_of(d); gc = d.set_index("brand").value[TARGET]; bo = best_other(d)
g, lo, hi, nn = paired("ALL", "mention", TARGET, bo.brand)
# title
ttl = f"{TARGET} unbranded search visibaility"
# subtitle
sub = "% of unbranded AI search answers mentioning each brand"


show(hbar_panels({STAGE_LABEL[s]: stage_board(s, "mention") for s in UNBRANDED}, 
                 pct, ttl, sub, note=with_missing(BAR_NOTE, missing_note(d.brand)), order_by="mean"), "1a_mention_by_section")

FINDINGS.append(("1 Share of voice", f"Named in {pct(gc)} of answers, #{r} of {n}. Gap vs {bo.brand}: {pts(g)} ({pts(lo)} to {pts(hi)}).", grade(lo, hi, nn, b=bo.brand), "Unbranded questions only; question set is ours"))

In [35]:
panels = {eng_name(e): board(e, "mention") for e in engines}
rk = {e: rank_of(board(e, "mention")) for e in engines}
ttl = f"{TARGET} unbranded search visibility across AI engines"
miss = missing_note(set().union(*[set(p.brand) for p in panels.values()]))
show(hbar_panels(panels, pct, ttl, "Share of answers naming each brand, by engine (average of the three question types)", note=with_missing(BAR_NOTE, miss), order_by="mean"), "1b_mention_by_engine")

## 4. Selection among mentions

**Question:** assess whether brands that are surfaced in unbranded AI answers are also identified as the top pick.

**Measured as:** top-pick rate ÷ mention rate. This measures how often a brand is identified as the top pick when it is mentioned. A response that lists a brand without identifying it as the top pick does not count as a selection.


In [36]:
# Retrieve metrics from boards
bm = board("ALL", "mention").set_index("brand")
bt = board("ALL", "top_pick").set_index("brand")
fb = board("ALL", "funnel").query("n >= @MIN_N").set_index("brand")

gcm = bm.value.get(TARGET, np.nan)
gcp = bt.value.get(TARGET, np.nan)
gcf = fb.value.get(TARGET, np.nan)

ttl = "Brand mention and top-pick rates"
sub = "For unbranded search queries"
fig = go.Figure()

# Main comparison set only
valid = [b for b in BRANDS_7 if b in bm.index and b in bt.index]

xvals = [bm.value[b] for b in valid] if valid else [0.05]
yvals = [bt.value[b] for b in valid] if valid else [0.05]

xmax = max(max(xvals) * 1.15, 0.05)
ymax = max(max(yvals) * 1.20, 0.05)

# Both axes start strictly at zero
xmin = 0.0
ymin = 0.0

n_max = fb.n.max() if len(fb) else 1

for b in valid:
    isg = b == TARGET

    n = fb.loc[b, "n"] if b in fb.index else np.nan

    # Dynamic circle sizing based on sample count
    size = (
        11
        if np.isnan(n)
        else max(11, 9 + 3 * np.sqrt(n / max(n_max, 1)))
    )

    y_val = bt.value[b]
    # Place text label above the dot if at y=0 so text does not cross below the axis
    # textpos = "top right" if y_val <= 0.005 else "middle right"
    textpos = "top center"

    fig.add_trace(go.Scatter(
        x=[bm.value[b]],
        y=[y_val],
        mode="markers+text",
        text=[f"<b>{b}</b>" if isg else b],
        textposition=textpos,
        cliponaxis=False,  # Prevents markers and labels at y=0 from being clipped by the x-axis
        marker=dict(
            size=size,
            color=color(b),
            line=dict(width=0)
        ),
        textfont=dict(
            size=11,
            color=GREEN if isg else "#555"
        ),
        hovertemplate=(
            f"{b}<br>"
            "Named: %{x:.0%}<br>"
            "Selected: %{y:.0%}<br>"
            f"Selected among mentions: "
            f"{pct(fb.value[b]) if b in fb.index else 'n/a'}"
            f"<br>n: {int(n) if not np.isnan(n) else 'n/a'}"
            "<extra></extra>"
        )
    ))

fig.update_xaxes(
    title="Brand mention rate",
    tickformat=".0%",
    showgrid=False,
    zeroline=False,
    range=[0, xmax]
)

fig.update_yaxes(
    title="Brand selection rate",
    tickformat=".0%",
    showgrid=False,
    zeroline=False,
    range=[0, ymax]
)

# Force the layout to be more square
fig.update_layout(
    width=650,
    height=650
)

# Render completed figure
show(
    _finish(
        fig,
        ttl,
        sub,
        height=650,
        note=missing_note(valid)
    ),
    "2a_funnel_scatter"
)

_f = OV[("ALL", TARGET, "funnel")]
FINDINGS.append((
    "2 Funnel conversion",
    f"{TARGET} named in {pct(gcm)} of answers and top pick in {pct(gcp)}; selected in {pct(gcf)} of the answers that name it"
    + (f" ({pct(_f[1])} to {pct(_f[2])})." if not np.isnan(_f[1]) else "."),
    "directional" if _f[3] >= MIN_N else "low (too few answers)",
    "Top-pick rate ÷ mention rate; unbranded questions only"
))

## 5. Head-to-head comparisons

**Question:** assess whether Good Culture is preferred over incumbent brands when the two brands are evaluated directly or surfaced together.

**Measured as:** two views of competitive comparison:

* **A. Named matchups:** direct "X vs. Y" questions, asked in both naming orders so the order of the brands in the prompt does not determine the result.
* **B. Implicit matchups:** unbranded answers that mention both Good Culture and another brand. The outcome is based on how the engine treated each brand: pick > recommended > listed > discouraged.

**Win rate:** wins ÷ (wins + losses + ties).


In [39]:
COLOR_WIN  = GREEN
COLOR_TIE  = GREY
COLOR_LOSS = "#C62828"

TEXT_WIN  = "#FFFFFF"
TEXT_TIE  = "#334155"
TEXT_LOSS = "#FFFFFF"


H = (
    J[J.type == "head_to_head"]
    .drop_duplicates("run_id")
    .merge(runs[["run_id", "prompt_id", "swapped"]], on="run_id")
)

H["rival"] = (
    H.prompt_id
    .str.replace("__swapped", "", regex=False)
    .map(lambda p: BRAND_BY_SLUG.get(p.split("__")[-1], p.split("__")[-1]))
)

# Main comparison set only
H = H[H.rival.isin(BRANDS_7)]

# Normalize outcome relative to target brand perspective
H["res"] = np.where(
    H.winner == TARGET,
    "Win",
    np.where(
        H.winner == H.rival,
        "Loss",
        "Tie / Inconclusive"
    )
)

# Collapse prompt orderings
tab = (
    H.groupby("rival")
    .res.value_counts()
    .unstack()
    .reindex(columns=["Win", "Tie / Inconclusive", "Loss"], fill_value=0)
    .fillna(0)
    .astype(int)
)

tab["Total Evals"] = tab.sum(axis=1)
tab["Win Rate"] = tab["Win"] / tab["Total Evals"]
tab["Tie Rate"] = tab["Tie / Inconclusive"] / tab["Total Evals"]
tab["Loss Rate"] = tab["Loss"] / tab["Total Evals"]

# Sort ascending
rivals_sorted = tab.sort_values(by=["Win Rate", "Tie Rate"], ascending=True).index.tolist()

h2h_missing = missing_note([TARGET] + list(tab.index), reason="no head-to-head prompt")
tab.reset_index().to_csv(TABLES / "head_to_head_summary.csv", index=False)


fig = go.Figure()

outcome_layers = [
    ("Win", COLOR_WIN, TEXT_WIN),
    ("Tie / Inconclusive", COLOR_TIE, TEXT_TIE),
    ("Loss", COLOR_LOSS, TEXT_LOSS)
]

for res_name, bg_col, text_col in outcome_layers:
    x_shares = []
    y_labels = []
    bar_texts = []

    for rival in rivals_sorted:
        total = tab.loc[rival, "Total Evals"]
        count = tab.loc[rival, res_name]
        share = count / total if total > 0 else 0

        x_shares.append(share)

        # No bolding: keep competitor labels neutral
        y_labels.append(f"vs. {rival}")

        # Percentages are shown directly in the bars
        bar_texts.append(pct(share) if share >= 0.08 else "")

    fig.add_trace(
        go.Bar(
            name=res_name,
            y=y_labels,
            x=x_shares,
            orientation="h",
            marker=dict(
                color=bg_col,
                line=dict(color="#FFFFFF", width=1.5)
            ),
            text=bar_texts,
            textposition="inside",
            insidetextanchor="middle",
            textfont=dict(
                color=text_col,
                size=12,
                family="Arial, Helvetica, sans-serif"
            ),
            hovertemplate=(
                "<b>%{y}</b><br>"
                + f"{res_name}: "
                + "%{x:.1%}<extra></extra>"
            )
        )
    )


total_evals = int(tab["Total Evals"].sum())
overall_win_rate = tab["Win"].sum() / total_evals
zero_loss_rivals = int((tab["Loss"] == 0).sum())
total_rivals = len(tab)

ttl = "Good Culture Head-to-head outcomes vs. named competitors"
sub = (
    f"Observed model preference across {total_rivals} competitors "
    f"(n={total_evals})"
)

fig = _finish(
    fig,
    title=ttl,
    subtitle=sub,
    note=h2h_missing,
    height=160 + 70 * total_rivals,
    width=800,
    top=125,   # more room for title/subtitle
)

fig.update_layout(
    title_y=0.95,
    title_yanchor="top",

    barmode="stack",
    bargap=0.35,
    showlegend=True,
    legend=dict(
        orientation="h",
        y=1.03,
        x=0,
        yanchor="bottom",
        font=dict(size=12, color="#333"),
        traceorder="normal"
    ),
    margin=dict(
        l=150,
        r=30,
        t=145,   # prevent title/subtitle cutoff
        b=70 if h2h_missing else 30
    )
)

fig.update_xaxes(
    title=None,
    tickformat=None,
    showticklabels=False,
    ticks="",                # removes the tick marks
    showline=False,          # removes the axis base line
    range=[0, 1.001],
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    showgrid=False,
    showline=False,
    ticks="",
    tickfont=dict(
        size=12,
        color="#1F2937"
    )
)


show(fig, "3a_head_to_head")
display(
    tab[["Win", "Tie / Inconclusive", "Loss", "Total Evals", "Win Rate", "Tie Rate", "Loss Rate"]]
)

FINDINGS.append(
    (
        "3 Head-to-head",
        (
            f"Good Culture was selected in {pct(overall_win_rate)} of direct competitive "
            f"evaluations; ties/inconclusive responses are shown separately across "
            f"{total_rivals} named competitors."
        ),
        "?",
        f"n={total_evals} evaluations across {total_rivals} named competitors; both prompt orders included"
    )
)

res,Win,Tie / Inconclusive,Loss,Total Evals,Win Rate,Tie Rate,Loss Rate
rival,,,,,,,
Breakstone's,4,2,0,6,0.666667,0.333333,0.0
Daisy,1,5,0,6,0.166667,0.833333,0.0
Knudsen,5,1,0,6,0.833333,0.166667,0.0


## 6. Positioning by attribute and shopper segment

**Question:** assess which product attributes and shopper segments are most strongly associated with Good Culture being identified as the top pick.

**Measured as:** % of responses (all LLMs and repeats) that identify each brand as the single top pick for each product attribute or shopper segment. Darker cells indicate higher selection rates. Each cell is based on approximately six responses, so patterns are more informative than small differences between percentages.


In [44]:
BRANDS_7 = ["Good Culture", "Daisy", "Hood", "Nancy's", "Breakstone's", "Lactaid", "Knudsen"]
Q = ELIG[ELIG.stage.isin(["attributes", "personas"])]
qcols = list(dict.fromkeys(Q.prompt_id))

available_brands = [b for b in BRANDS_7 if b in VIS]

_pick_src = pd.DataFrame({
    "prompt_id": ELIG["prompt_id"].values,
    "stage":     ELIG["stage"].values,
    **{b: NUMM[(b, "top_pick")].values for b in available_brands}
})
_pick_src = _pick_src[_pick_src["stage"].isin(["attributes", "personas"])].copy()

pick = pd.DataFrame({
    b: _pick_src.groupby("prompt_id")[b].mean().reindex(qcols)
    for b in available_brands
}).T

# Attributes to exclude from the graph
exclude_attrs = ["live cultures", "all of our positioning", "lactose free", "sodium"]

attr_cols = [
    p for p in qcols
    if Q.loc[Q.prompt_id == p, "stage"].iloc[0] == "attributes"
    # Filter out by both raw prompt ID and the 'pretty' version
    and str(p).lower() not in exclude_attrs
    and pretty(p).lower() not in exclude_attrs
]

segment_cols = [
    p for p in qcols
    if Q.loc[Q.prompt_id == p, "stage"].iloc[0] == "personas"
]


def make_matrix(cols, title, subtitle, save_name):
    m = pick[cols].copy()

    # Order brands by average selection within this table
    avg_scores = m.mean(axis=1).sort_values(ascending=False)
    
    # Force TARGET to the front of the list, followed by the rest
    sorted_brands = [TARGET] + [b for b in avg_scores.index if b != TARGET]
    
    # Apply the new order (filtering just in case TARGET isn't in the index)
    m = m.loc[[b for b in sorted_brands if b in m.index]]

    def label(p):
        val = pretty(p)
        
        # Apply requested string replacements
        if val.lower() == "taste texture":
            val = "taste/texture"
        elif val.lower() == "price value":
            val = "price/value"
            
        return "<br>".join(
            textwrap.wrap(val, 16)
        )

    xl = [label(p) for p in cols]

    Z = m.values
    text = [
        [f"{v:.0%}" if v > 0 else "" for v in row]
        for row in Z
    ]

    rows_lab = [
        f"<b style='color:{GREEN}'>{b}</b>" if b == TARGET else b
        for b in m.index
    ]

    fig = go.Figure(
        go.Heatmap(
            z=Z,
            x=xl,
            y=rows_lab,
            text=text,
            texttemplate="%{text}",
            colorscale=[
                [0, "#F6F7F9"],
                [1, "#8C97A3"]
            ],
            zmin=0,
            zmax=max(0.6, np.nanmax(Z)),
            showscale=False,
            xgap=2,
            ygap=2,
            hovertemplate="%{y} — %{x}: %{z:.0%}<extra></extra>"
        )
    )

    if TARGET in m.index:
        target_row = list(m.index).index(TARGET)

        for col_idx in range(len(cols)):
            fig.add_shape(
                type="rect",
                x0=col_idx - 0.5,
                x1=col_idx + 0.5,
                y0=target_row - 0.5,
                y1=target_row + 0.5,
                fillcolor="rgba(0,0,0,0)",
                line=dict(color=GREEN, width=2),
                layer="above"
            )

    fig.update_xaxes(
        side="top",
        showgrid=False,
        showline=False,
        ticks="",
        tickfont=dict(size=11)
    )

    fig.update_yaxes(
        autorange="reversed",
        showgrid=False,
        showline=False,
        ticks=""
    )

    # Pass the standard title into the chart visual, but use save_name for the file/output name
    show(
        _finish(
            fig,
            title, 
            subtitle,
            with_missing("Percentages are raw top-pick rates; they do not sum to 100%.", missing_note(m.index)),
            125 + 34 * len(m),
            top=135
        ),
        save_name
    )

    return m


attr_pick = make_matrix(
    attr_cols,
    "Brand selection - product attributes",
    "% of responses selecting each brand for the stated product attribute.",
    "4a_brand_selection_product_attributes"
)

segment_pick = make_matrix(
    segment_cols,
    "Brand selection - shopper segments",
    "% of responses selecting each brand for the stated shopper segment.",
    "4b_brand_selection_shopper_segments"
)

# Exporting the underlying tables with matching 4a/4b naming
attr_pick.to_csv(TABLES / "brand_positioning_product_attributes.csv")
segment_pick.to_csv(TABLES / "brand_positioning_shopper_segments.csv")

## 7. Tone: how do engines talk about each brand?
**Question:** a brand can be named often and still be described lukewarmly. Tone shows whether engines frame Good Culture as a strong choice or just an option.
**Measured as:** the AI judge scores the tone of each answer toward each named brand from −2 (negative) to +2 (positive); we average it over the answers that name the brand, in unbranded questions. It says nothing about brands that are never named. Very few answers are negative in practice, so differences are small and ranges wide.

In [42]:
d = board("ALL", "sentiment").reset_index(drop=True)
ok = d[d.n >= MIN_N]

if TARGET in d.brand.values and len(ok):
    gc = d.set_index("brand").value[TARGET]
    r, n_br = rank_of(d)

    ttl = "Average tone when brand is named"
    sub = "AI-judged tone when each brand is named; 0 = neutral, +2 = strongly positive"

    fig = hbar_panels(
        {"Average tone when named": d},
        lambda v: f"{v:+.1f}",
        ttl,
        sub,
        note=with_missing(BAR_NOTE, missing_note(d.brand, default="no tone scores")),
        top_n=len(d),
        spacing=.1,
        xtickfmt=".1f"
    )

    # 0–2 axis with room for the value labels
    fig.update_xaxes(range=[0, 2.4], tickvals=[0, 0.5, 1, 1.5, 2], tickformat=".1f", title=None)

    show(fig, "5_average_tone")

    # Sample sizes retained in the underlying table, not on the chart face
    d.to_csv(TABLES / "average_tone_by_brand.csv", index=False)

    FINDINGS.append((
        "Tone",
        f"Average tone {gc:+.1f} for {TARGET} across the seven-brand comparison set "
        f"(#{r} of {n_br}); n varies by brand.",
        "directional",
        "Judge scale −2 to +2, observed scores non-negative; engines rarely criticise a brand they name"
    ))

else:
    print("Not enough scored mentions of the target to chart tone.")

## 8. Cross-engine consistency

**Question:** assess whether Good Culture's relative top-pick rate is consistent across AI abbwers or varie.

**Measured as:** Good Culture's top-pick rate minus the top-pick rate of the strongest other brand on each engine. The comparison is made separately by question type and overall. Positive values indicate a higher Good Culture top-pick rate; negative values indicate a higher rate for another brand. `*` indicates that the 95% bootstrap interval does not include zero.

In [11]:
gc_t = COLIX[(TARGET, "top_pick")]; oth_names = [b for b in BRANDS_CMP if b != TARGET]; oth_t = [COLIX[(b, "top_pick")] for b in oth_names]
def margin(eng, stage):
    dr = STORE[(eng, stage)]; pt = POINT[(eng, stage)]
    o = pt[oth_t]; j = int(np.nanargmax(o)); g = pt[gc_t] - o[j]
    dd = dr[:, CIPOS[gc_t]] - np.nanmax(dr[:, [CIPOS[c] for c in oth_t]], axis=1)
    n = NROWS[(eng, stage)]; lo, hi = rng95(dd) if n >= MIN_N else (np.nan, np.nan)
    return g, lo, hi, oth_names[j], n
def margin_overall(eng):
    pt = ov_point(eng, [gc_t] + oth_t); o = pt[1:]; j = int(np.nanargmax(o))
    dd = ov_draws(eng, [gc_t])[:, 0] - np.nanmax(ov_draws(eng, oth_t), axis=1)
    n = sum(NROWS[(eng, s)] for s in cells(eng)); lo, hi = rng95(dd) if n >= MIN_N else (np.nan, np.nan)
    return pt[0] - o[j], lo, hi, oth_names[j], n
rowsM = {}
for s in UNBRANDED: rowsM[STAGE_LABEL[s]] = {e: margin(e, s) for e in engines}
rowsM["Overall"] = {e: margin_overall(e) for e in engines}
ylab = list(rowsM); Z = np.array([[rowsM[y][e][0] for e in engines] for y in ylab]) * 100
txt = [[f"<b>{rowsM[y][e][0]*100:+.0f} pts</b>{'*' if (not np.isnan(rowsM[y][e][1]) and (rowsM[y][e][1] > 0 or rowsM[y][e][2] < 0)) else ''}<br><span style='font-size:10px'>vs. {rowsM[y][e][3]}</span>" for e in engines] for y in ylab]
lim = max(30, np.nanmax(np.abs(Z)))
fig = go.Figure(go.Heatmap(z=Z, x=[eng_name(e) for e in engines], y=ylab, text=txt, texttemplate="%{text}", zmid=0, zmin=-lim, zmax=lim,
                           colorscale=[[0, "#8C97A3"], [0.5, "#FFFFFF"], [1, GREEN]], showscale=False, xgap=3, ygap=3))
ov_ = rowsM["Overall"]; ahead = [e for e in engines if ov_[e][0] > 0]; clear = [e for e in engines if not np.isnan(ov_[e][1]) and ov_[e][1] > 0]
ttl = "Good Culture top-pick rate vs. strongest other brand"
fig.update_xaxes(side="top", showgrid=False, showline=False, ticks="", tickfont=dict(size=13)); fig.update_yaxes(autorange="reversed", showgrid=False, showline=False, ticks="", tickfont=dict(size=13))
show(_finish(fig, ttl, "Difference in top-pick rate by engine and question type; positive = Good Culture higher", missing_note(BRANDS_CMP), 330, top=130), "6_engine_consistency")
FINDINGS.append((
    "6 Cross-engine consistency",
    "Top-pick rate gap vs. strongest other brand: "
    + "; ".join(f"{eng_name(e)} {ov_[e][0]*100:+.0f} pts (vs. {ov_[e][3]})" for e in engines)
    + f". Good Culture has a positive margin on {len(ahead)} of {len(engines)} engines "
      f"(95% interval excludes zero on {len(clear)}).",
    "Cross-engine consistency of brand selection",
    "Margins vs. strongest other brand; comparator selected separately by engine"
))

## 9. Citation source mix

**Question:** assess whether Good Culture is surfaced alongside citations from independent third-party sources or primarily from the brand's own website.

**Measured as:** the types of sources cited in AI answers that mention each brand, grouped into third-party sources and brand-owned sources. This indicates the source environment surrounding brand mentions; it does not establish whether a citation caused the brand to be mentioned or whether the cited source specifically supports the brand.


In [12]:
if C.empty:
    print("no citations.csv found")
else:
    Cc = C[
        ~C.domain_type.isin(["google_shopping", "search_page", "unclassified"])
    ].merge(
        runs[["run_id", "stage", "group"]],
        on="run_id"
    )

    # Core, unbranded questions only
    Cc = Cc[
        (Cc.group == "core") &
        Cc.stage.isin(UNBRANDED)
    ].copy()

    # Brands present in each answer
    pres_df = pd.DataFrame(
        list(present_sets(ELIG)),
        columns=["run_id", "brand"]
    )

    cb = Cc.merge(pres_df, on="run_id")

    cb = cb[cb.brand.isin(BRANDS_7)].copy()

    # Classify citation source
    own = cb.domain.map(OWNED_DOMAINS)

    cb["kind"] = np.select(
        [
            own == cb.brand,
            own.notna(),
            cb.domain_type == "retailer",
            cb.domain_type.isin(["editorial", "health_authority", "community"])
        ],
        [
            "First-party (brand site)",
            "Another brand's site",
            "Retailer",
            "Third-party"
        ],
        default="Other"
    )

    # Save underlying table
    cb.groupby(
        ["kind", "brand", "engine"]
    ).size().rename("citations").reset_index().to_csv(
        TABLES / "citations_by_brand.csv",
        index=False
    )

    KINDS = [
        "First-party (brand site)",
        "Retailer",
        "Third-party",
        "Another brand's site",
        "Other"
    ]

    mix = (
        cb.groupby(["brand", "kind"])
        .size()
        .unstack()
        .reindex(index=BRANDS_7, columns=KINDS)
        .fillna(0)
    )

    mix["n"] = mix.sum(axis=1)

    # Convert citation counts to within-brand shares
    sh = mix[KINDS].div(mix["n"], axis=0)

    tp = pd.DataFrame({
        "brand": sh.index,
        "value": sh["Third-party"].values,
        "n":     mix["n"].values,
        "flag":  ["*" if x < MIN_N else "" for x in mix["n"].values]
    })

    # No CI computed for citation shares -> collapse error bars to the point estimate
    tp["lo"] = tp["value"]
    tp["hi"] = tp["value"]

    tp["brand"] = tp["brand"].astype(str)
    tp_ok = tp.dropna(subset=["value"])

    if TARGET in tp_ok.brand.values:

        gct = tp_ok.set_index("brand").value[TARGET]

        note = with_missing(
            "Third-party = editorial, health-authority or community sources. * = fewer than 8 citations.",
            missing_note(tp_ok.brand, default="no cited sources")
        ).replace(" Citations share", "<br>Citations share")
        fig = hbar_panels(
            {"Third-party share of cited sources": tp_ok},
            pct,
            "Citation source mix when brands are mentioned",
            "Share of cited sources that are third-party vs. brand-owned",
            note=note,
            top_n=len(BRANDS_7),
            spacing=.1
        )

        show(fig, "7a_third_party_share")

        FINDINGS.append((
            "7 Citations",
            f"{TARGET}: {pct(gct)} of cited sources were classified as third-party.",
            "Source environment surrounding brand mentions",
            "Citation is in the same answer, not necessarily about or supporting the brand."
        ))

    print(
        f"{(C.domain_type == 'unclassified').mean():.0%} of citations are still "
        "unclassified and were skipped (classify them in learned.yaml and rerun parse)."
    )

0% of citations are still unclassified and were skipped (classify them in learned.yaml and rerun parse).


## 12. Summary

In [13]:
out = ["| Test | What we found | Confidence | Caution |", "|---|---|---|---|"]
for a, b, c, d in FINDINGS: out.append(f"| {a} | {b} | {c} | {d} |")
display(Markdown("\n".join(out)))


| Test | What we found | Confidence | Caution |
|---|---|---|---|
| 1 Share of voice | Named in 81% of answers, #1 of 6. Gap vs Daisy: +17 pts (+1 pts to +34 pts). | high confidence | Unbranded questions only; question set is ours |
| 2 Funnel conversion | Good Culture named in 81% of answers and top pick in 27%; selected in 34% of the answers that name it (16% to 51%). | directional | Top-pick rate ÷ mention rate; unbranded questions only |
| 3 Head-to-head | Good Culture was selected in 56% of direct competitive evaluations; ties/inconclusive responses are shown separately across 3 named competitors. | ? | n=18 evaluations across 3 named competitors; both prompt orders included |
| Tone | Average tone +1.4 for Good Culture across the seven-brand comparison set (#1 of 6); n varies by brand. | directional | Judge scale −2 to +2, observed scores non-negative; engines rarely criticise a brand they name |
| 6 Cross-engine consistency | Top-pick rate gap vs. strongest other brand: ChatGPT +16 pts (vs. Daisy); Gemini +2 pts (vs. Daisy); Perplexity +27 pts (vs. Daisy). Good Culture has a positive margin on 3 of 3 engines (95% interval excludes zero on 0). | Cross-engine consistency of brand selection | Margins vs. strongest other brand; comparator selected separately by engine |
| 7 Citations | Good Culture: 52% of cited sources were classified as third-party. | Source environment surrounding brand mentions | Citation is in the same answer, not necessarily about or supporting the brand. |

In [47]:
board("ALL", "top_pick")
# board("ALL", "funnel")

,brand,value,lo,hi,n,flag
0,Good Culture,0.271605,0.124691,0.435802,114,
1,Daisy,0.120988,0.024691,0.234599,114,
2,Hood,0.044444,0.000000,0.111111,114,
3,Nancy's,0.012346,0.000000,0.049383,114,
4,Breakstone's,0.000000,0.000000,0.000000,114,
5,Lactaid,0.000000,0.000000,0.000000,114,
